# MP7: Few-Shot Trope/Genre Classification
Genesis Joseph, CIS 731. The protocol is in `protocol.md`, committed before this notebook was run.

This notebook runs on CPU only. The first run downloads `all-MiniLM-L6-v2` (about 90 MB) and `Qwen2.5-0.5B-Instruct` (about 1 GB).

In [ ]:
import json, sys
from pathlib import Path
sys.path.insert(0, '.'); sys.path.insert(0, '../MP6_Genesis/schema')
import mp7lib as M, run_mp7 as R
print('labels:', M.LABEL_IDS)
print('ks:', R.KS, 'seeds:', R.SEEDS)

## 1. Run the sweep
This step admits the exemplars through the PS6 contracts, builds the three classifiers and runs k ∈ {0,1,3,5,8} × 5 seeds. It writes `predictions.jsonl`, `results/` and `report.md`.

In [ ]:
summary, best, per_class, run_log = R.main()

## 2. Results audit: re-derive every reported number from `predictions.jsonl`

In [ ]:
rows = M.load_jsonl('predictions.jsonl')
eval_ids = [r['record_id'] for r in M.load_jsonl('data/eval_mp7.jsonl')]
audit = M.summarize(rows, eval_ids, B=1000)
for a, s in zip(audit, summary):
    assert (a['classifier'], a['k']) == (s['classifier'], s['k']) and abs(a['mean'] - s['mean']) < 1e-12
print(f"{'classifier':10s} {'k':>2s}  mean    std     95% CI")
for a in audit:
    print(f"{a['classifier']:10s} {a['k']:2d}  {a['mean']:.3f}  {a['std']:.3f}  [{a['ci_lo']:.3f}, {a['ci_hi']:.3f}]")
print('\nall means reproduced from predictions.jsonl')

## 3. Contract checks on every stored probability row

In [ ]:
import numpy as np
P = np.array([[r['proba'][l] for l in M.LABEL_IDS] for r in rows])
print('rows:', len(P), '| min prob:', P.min(), '| max |row sum - 1|:', np.abs(P.sum(1) - 1).max())
assert (P >= 0).all() and np.abs(P.sum(1) - 1).max() <= 1e-5   # stored values are rounded to 6 decimals
assert all(r['pred'] in M.LABEL_IDS for r in rows)
print('all labels in the declared label space')

## 4. Failure-mode controls: evidence

In [ ]:
from collections import Counter
llm = [r for r in rows if r['classifier'] == 'llm' and r['k'] == 3]
orders = {r['seed']: r['demo_ids'] for r in llm}
print('demo order differs across seeds:', len({tuple(v) for v in orders.values()}) == len(orders))
label_of = {r['record_id']: r['label'] for r in M.load_jsonl('data/exemplars.jsonl')}
print('class balance (k=3, seed 0):', Counter(label_of[i] for i in orders[0]))
print('eval/demo overlap:', set(eval_ids) & {i for r in rows for i in r['demo_ids']})

## 5. Best configuration: per-class precision and recall, plus the run log

In [ ]:
print('best:', best['classifier'], 'k =', best['k'], f"macro-F1 = {best['mean']:.3f}")
for lbl, (p, r) in per_class.items():
    print(f'{lbl:12s} P={p:.2f} R={r:.2f}')
print(json.dumps(run_log, indent=2))